# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use Croissant `@id` fields to reference record sets and fields.

In [ ]:
# List all record sets by their @id and name
print("Available Record Sets:")
record_sets = []
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id} | name: {rs.name}")
    record_sets.append(rs.id)

# Show fields for each record set
print("\nFields in each Record Set:")
for rs in metadata.record_sets:
    print(f"\nRecordSet @id: {rs.id} -- name: {rs.name}")
    for f in rs.fields:
        print(f"    Field @id: {f.id}, name: {f.name}, dataType: {f.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from one or more record sets
# (Replace the `main_record_set_id` below with the @id of your primary record set. We'll extract all record sets here.)

dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No data loaded for {record_set_id}.")
    print("-"*40)

# For the remainder of this notebook, we'll select the first non-empty record set for EDA
for rid in record_sets:
    if not dataframes[rid].empty:
        main_record_set_id = rid
        break

print(f"\nSelected main RecordSet for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All field references use the Croissant field `@id`.

In [ ]:
# Identify numeric fields by their Croissant field @id in the selected record set
main_rs = None
for rs in metadata.record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break
numeric_fields = [f.id for f in main_rs.fields if f.data_type in ('Integer','Float','Number','schema:Float','schema:Integer','schema:Number')]

print(f"Numeric field IDs: {numeric_fields}")
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise ValueError("No numeric fields found.")

df = dataframes[main_record_set_id]
# Make sure to convert the field to numeric, coerce errors
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering: Keep rows with value > threshold
threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 10
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
# Find a categorical field (e.g., protein description, etc)
cat_fields = [f.id for f in main_rs.fields if f.data_type in ('Text','schema:Text') and f.id != numeric_field_id]
if cat_fields:
    group_field_id = cat_fields[0]
    print(f"\nGrouping by: {group_field_id}")
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_").reset_index()
    )
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouped_df is available, plot group means
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 4))
    # Plot top categories (if too many)
    topg = grouped_df.sort_values('mean_', ascending=False).head(10)
    sns.barplot(y=group_field_id, x='mean_', data=topg)
    plt.title(f"Mean of {numeric_field_id} by {group_field_id} (top 10)")
    plt.xlabel(numeric_field_id)
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We explored the FAIR² dataset describing mass spectrometry analysis of extracellular vesicles from human mast cells.
- Loaded its schema and metadata with `mlcroissant`, identified available record sets and fields using their `@id`s.
- Extracted and analyzed the main record set, filtered and normalized a numeric field, and grouped by a categorical attribute.
- Visualizations revealed data distributions and highlighted differences by group, demonstrating how Croissant metadata streamlines FAIR data loading and exploration.

**Next steps could include machine learning analysis or harmonized integration with similar Croissant-packaged datasets.**